## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


The pipeline version is pinned (-r), every tool runs in a container with a fixed version, parameters and references are recorded in pipeline_info/, the input comes from public SRA accessions, and the workflow is standardised, tested and kept in Git. Anyone gets the same results.

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

nf-core/rnaseq

It takes raw RNA-seq FASTQ files and performs quality control and RNA-seq processing, ultimately producing gene/transcript-level quantification/counts.

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

The pipeline performs the major preprocessing steps needed for bulk RNA-seq:

Checks the quality of raw reads.
Detects/removes adapter or low-quality sequence when appropriate.
Aligns or pseudo-aligns reads to the reference.
Produces alignment/quantification information.
Generates gene/transcript count tables.
Performs extensive QC and combines the results into a MultiQC report.
Main tools

Depending on the pipeline configuration, important tools include:

FastQC → raw-read quality
Trim Galore / Cutadapt → adapter/quality trimming
STAR / Salmon → alignment or transcript quantification
SAMtools → processing alignment files
featureCounts → gene-level counting
RSeQC → RNA-seq-specific QC
MultiQC → combines QC results into one report

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [2]:
import pandas as pd

samplesheet = pd.DataFrame({
    "sample": [
        "Oxy_SNI_SRR23195516",
        "Oxy_Sham_SRR23195511"
    ],
    "fastq_1": [
        "fetchngs_out/fastq/SRX19144486_SRR23195516_1.fastq.gz",
        "fetchngs_out/fastq/SRX19144488_SRR23195511_1.fastq.gz"
    ],
    "fastq_2": [
        "fetchngs_out/fastq/SRX19144486_SRR23195516_2.fastq.gz",
        "fetchngs_out/fastq/SRX19144488_SRR23195511_2.fastq.gz"
    ],
    "strandedness": [
        "auto",
        "auto"
    ]
})

samplesheet.to_csv("samplesheet.csv", index=False)

samplesheet

,sample,fastq_1,fastq_2,strandedness
0,Oxy_SNI_SRR23195516,fetchngs_out/fastq/SRX19144486_SRR23195516_1.f...,fetchngs_out/fastq/SRX19144486_SRR23195516_2.f...,auto
1,Oxy_Sham_SRR23195511,fetchngs_out/fastq/SRX19144488_SRR23195511_1.f...,fetchngs_out/fastq/SRX19144488_SRR23195511_2.f...,auto


In [11]:
import os
os.environ["PATH"] += os.pathsep + "/Applications/Docker.app/Contents/Resources/bin"
!docker ps

!pwd

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES
/Users/km/Documents/GitHub/computational-workflows-2026/day_02


In [ ]:
# post here the command you used to run nf-core/rnaseq

!nextflow run nf-core/rnaseq \
  -r 3.27.0 \
  -profile docker \
  --input samplesheet.csv \
  --outdir rnaseq_results \
  --fasta reference/chr1.fa \
  --gtf reference/chr1.gtf

Explain all the parameters you set and why you set them in this way.



-r: fixed version.
docker: containers.
--input / --outdir: samplesheet and output folder.
--fasta / --gtf: mouse reference, from the same release.

## Browsing the results

How did the pipeline perform?

Overall, the pipeline performed successfully and produced QC, trimming, alignment and downstream sample-level analyses for 16 samples. The raw reads were generally of good sequencing quality, and trimming substantially improved the QC profile. However, there are some sample-specific problems, particularly in alignment/mapping and GC-content profiles.     MultiQC Report
The biggest concern is SRX19144491, which has only 28.2% uniquely aligned reads and 26.1% total aligned reads. Other relatively low-mapping samples include SRX19144490 (59.5% unique) and SRX19144489 (65.2% unique). In contrast, several samples have >90% unique alignment.

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication
- GC content

What are the possible steps that could lead to poorer results?

Quality control steps -
FastQC → sequencing quality
Trim Galore → adapter/quality trimming
STAR → genome alignment
Picard/Samtools → alignment & duplication QC
RSeQC/Qualimap → RNA-seq-specific QC
featureCounts → read counting
DESeq2 → PCA and sample clustering

Ribosomal rRNA?
The rRNA fraction is relatively small, while most reads are assigned to protein-coding genes. Therefore, rRNA contamination does not appear to be a major issue.  
Duplication?
Some duplication is present, but duplication alone is not necessarily problematic in RNA-seq, because highly expressed genes naturally produce many identical reads.    
GC content?
Some samples show deviations from the expected GC distribution, including high-GC peaks. This could indicate GC bias or unusual/overrepresented sequences and should be investigated.

Possible causes include:
- poor/degraded RNA
- contamination
- library-preparation bias
- low library complexity
- sequencing problems
- alignment/reference issues

Would you exclude any samples? If yes, which and why?

SRX19144491 is the strongest candidate because only 28.2% of reads uniquely aligned. SRX19144489 (65.2%) and SRX19144490 (59.5%) should also be investigated. Also, the candidates with high GC content could be excluded.

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

After QC:
Count matrix → normalization → PCA/clustering → differential expression → pathway enrichment
DESeq2 is used for differential expression and PCA.

Scientists want to identify genes whose expression differs between experimental conditions and determine which biological pathways/processes are affected.